# Experiment: Symbolic Music Benchmark Review

Objective:
- Review the project as a symbolic benchmark for music theory and computational musicology.
- Run a small, reproducible end-to-end benchmark path.
- Inspect the standard benchmark mode and the optional explanatory mode.

Success criteria:
- generate a small benchmark sample
- build model-ready cases
- inspect label and sequence tasks
- run an oracle-style evaluation sanity check
- review how `prediction_explanation` fits into the pipeline


In [1]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

SEED = 7


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'benchmark').exists():
            return candidate
    raise RuntimeError('Could not find repo root from current notebook location.')


NOTEBOOK_PATH = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_PATH)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmark.core.musicology import EXPLANATION_POLICY, THEORETICAL_SIMPLIFICATIONS
from benchmark.core.task_specs import ALL_TASKS, LABEL_TASKS, SEQUENCE_TASKS

WORKSPACE = REPO_ROOT / 'tmp' / 'notebook-benchmark-review'
WORKSPACE.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'seed': SEED,
    'samples_per_task': 4,
    'prompt_mode': 'agent_like',
    'workspace': str(WORKSPACE),
}
CONFIG


{'seed': 7,
 'samples_per_task': 4,
 'prompt_mode': 'agent_like',
 'workspace': '/Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review'}

## Plan

This notebook follows the simplest benchmark path for non-expert users, while still making the scientific framing explicit.

- Standard mode:
  - required
  - easy to test
  - automatically evaluable
  - based on `prediction_label` and `prediction_notes`
- Explanatory mode:
  - optional
  - useful for paper writing and qualitative analysis
  - adds `prediction_explanation` for label tasks

We will use a small sample size so the workflow stays fast and easy to inspect.


In [2]:
# Inspect the benchmark structure
summary = {
    'all_tasks': ALL_TASKS,
    'label_tasks': sorted(LABEL_TASKS),
    'sequence_tasks': sorted(SEQUENCE_TASKS),
    'explanation_policy': EXPLANATION_POLICY,
    'theoretical_simplifications': THEORETICAL_SIMPLIFICATIONS,
}
summary


{'all_tasks': ['task1_interval_identification',
  'task2_chord_identification',
  'task3_harmonic_function',
  'task4_transposition',
  'task5_melodic_inversion',
  'task6_retrograde',
  'task7_rhythm_scale',
  'task8_voice_leading'],
 'label_tasks': ['task1_interval_identification',
  'task2_chord_identification',
  'task3_harmonic_function',
  'task8_voice_leading'],
 'sequence_tasks': ['task4_transposition',
  'task5_melodic_inversion',
  'task6_retrograde',
  'task7_rhythm_scale'],
 'explanation_policy': {'mode': 'hybrid',
  'summary': 'V1 uses rule-based reference explanations and allows optional model-generated prediction_explanation fields at inference time.'},
 'theoretical_simplifications': {'harmonic_function': ['major mode only in V1',
   'three coarse function classes: tonic, predominant, dominant'],
  'voice_leading': ['two-voice simplification in V1',
   'labels limited to parallel_fifths, voice_crossing, and none'],
  'harmony': ['reduced harmonic vocabulary in V1',
   '

## Standard benchmark workflow

The cells below implement the standard benchmark path:

1. generate benchmark data
2. export tokenizer views
3. validate tokenizer consistency
4. build model-ready cases
5. inspect cases
6. generate an oracle prediction file for sanity checking
7. evaluate and annotate outputs

If you only want a shell wrapper, you can also use:

```bash
bash scripts/run_standard_benchmark.sh 200 agent_like none
```


In [3]:
# Helper for running repo commands from the notebook

def run_cmd(cmd: list[str], cwd: Path = REPO_ROOT) -> str:
    print('>>>', ' '.join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        check=True,
    )
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    return proc.stdout


RAW_PATH = WORKSPACE / 'zero_shot.jsonl'
VIEWS_DIR = WORKSPACE / 'views'
CASES_PATH = WORKSPACE / 'all_cases.json'
PRED_PATH = WORKSPACE / 'oracle_predictions.jsonl'
EVAL_PATH = WORKSPACE / 'eval_predictions.json'
ANNOTATED_PATH = WORKSPACE / 'cases_with_outputs.json'
SUMMARY_PATH = WORKSPACE / 'annotated_summary.json'

for target in [VIEWS_DIR]:
    if target.exists():
        shutil.rmtree(target)
for target in [RAW_PATH, CASES_PATH, PRED_PATH, EVAL_PATH, ANNOTATED_PATH, SUMMARY_PATH]:
    if target.exists():
        target.unlink()


In [4]:
# Step 1: generate a small raw benchmark sample
run_cmd([
    sys.executable,
    '-m',
    'benchmark.scripts.gen_zero_shot',
    '--task-group',
    'all',
    '--samples-per-task',
    str(CONFIG['samples_per_task']),
    '--prompt-mode',
    CONFIG['prompt_mode'],
    '--out',
    str(RAW_PATH),
])


>>> /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/asmcm-env/bin/python -m benchmark.scripts.gen_zero_shot --task-group all --samples-per-task 4 --prompt-mode agent_like --out /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/zero_shot.jsonl
generated: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/zero_shot.jsonl
tasks: 8, samples/task: 4, total: 32


'generated: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/zero_shot.jsonl\ntasks: 8, samples/task: 4, total: 32\n'

In [5]:
# Step 2 and 3: export and validate tokenizer views
run_cmd([
    sys.executable,
    '-m',
    'benchmark.scripts.export_views',
    '--src',
    str(RAW_PATH),
    '--out-dir',
    str(VIEWS_DIR),
    '--prompt-mode',
    CONFIG['prompt_mode'],
])

run_cmd([
    sys.executable,
    '-m',
    'benchmark.scripts.validate_views',
    '--views-dir',
    str(VIEWS_DIR),
])


>>> /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/asmcm-env/bin/python -m benchmark.scripts.export_views --src /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/zero_shot.jsonl --out-dir /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/views --prompt-mode agent_like
exported: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/views/note_level/zero_shot.jsonl
exported: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/views/midilike/zero_shot.jsonl
exported: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/views/remilike/zero_shot.jsonl
>>> /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/asmcm-env/bin/python -m benchmark.scripts.validate_views --views-dir /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/views
OK: id/task alignment across 3 tokenizers
OK: sequence target decode consistency checked=16


'OK: id/task alignment across 3 tokenizers\nOK: sequence target decode consistency checked=16\n'

In [6]:
# Step 4: build model-ready cases for the note_level view
run_cmd([
    sys.executable,
    '-m',
    'benchmark.scripts.build_model_io_json',
    '--src',
    str(VIEWS_DIR / 'note_level' / 'zero_shot.jsonl'),
    '--out',
    str(CASES_PATH),
    '--task-group',
    'all',
    '--pretty',
])


>>> /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/asmcm-env/bin/python -m benchmark.scripts.build_model_io_json --src /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/views/note_level/zero_shot.jsonl --out /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/all_cases.json --task-group all --pretty
wrote: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/all_cases.json
cases: 32


'wrote: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/all_cases.json\ncases: 32\n'

In [7]:
# Step 5: inspect one label case and one sequence case
cases = json.loads(CASES_PATH.read_text(encoding='utf-8'))
label_case = next(case for case in cases if case['task'] in LABEL_TASKS)
sequence_case = next(case for case in cases if case['task'] in SEQUENCE_TASKS)

inspection = {
    'label_case': {
        'case_id': label_case['case_id'],
        'task': label_case['task'],
        'prediction_schema': label_case['prediction_schema'],
        'usage_modes': label_case['usage_modes'],
        'target_explanation': label_case.get('target_explanation', ''),
    },
    'sequence_case': {
        'case_id': sequence_case['case_id'],
        'task': sequence_case['task'],
        'prediction_schema': sequence_case['prediction_schema'],
        'usage_modes': sequence_case['usage_modes'],
    },
}
inspection


{'label_case': {'case_id': 'task1_interval_identification-000000',
  'task': 'task1_interval_identification',
  'prediction_schema': {'prediction_type': 'label',
   'primary_field': 'prediction_label',
   'optional_fields': ['prediction_explanation'],
   'fallback_fields': ['prediction']},
  'usage_modes': {'standard': {'required_prediction_fields': ['prediction_label'],
    'optional_prediction_fields': [],
    'description': 'Default benchmark mode for non-expert users. Focus on strict, automatically evaluable outputs.'},
   'explanatory': {'required_prediction_fields': ['prediction_label'],
    'optional_prediction_fields': ['prediction_explanation'],
    'description': 'Optional analysis mode for paper-oriented qualitative work. The main label or note prediction remains primary; prediction_explanation is supplementary.'}},
  'target_explanation': 'A#3 to C#4 spans 3 semitones, which corresponds to a minor third.'},
 'sequence_case': {'case_id': 'task4_transposition-000012',
  'task

## Standard vs explanatory modes

The benchmark is designed around two usage modes.

### Standard mode
- mandatory
- simple
- suitable for non-expert users
- evaluated automatically

### Explanatory mode
- optional
- used for paper writing and qualitative analysis
- adds `prediction_explanation` for label tasks
- does not replace the main prediction field


In [8]:
# Step 6: create an oracle prediction file
# This produces perfect predictions directly from the gold targets.

gold_rows = [
    json.loads(line)
    for line in (VIEWS_DIR / 'note_level' / 'zero_shot.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]
case_map = {case['case_id']: case for case in cases}

with PRED_PATH.open('w', encoding='utf-8') as f:
    for row in gold_rows:
        task = row['task']
        rec = {
            'id': row['id'],
            'prediction_type': 'label' if task in LABEL_TASKS else 'notes',
            'prediction': row['target'],
        }
        if task in LABEL_TASKS:
            rec['prediction_label'] = row['target']
            rec['prediction_explanation'] = case_map[row['id']].get('target_explanation', '')
        else:
            rec['prediction_notes'] = row['target']
            rec['prediction_structured'] = row['target_payload']
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

preview_rows = [json.loads(line) for line in PRED_PATH.read_text(encoding='utf-8').splitlines()[:2]]
preview_rows


[{'id': 'task1_interval_identification-000000',
  'prediction_type': 'label',
  'prediction': 'minor_third',
  'prediction_label': 'minor_third',
  'prediction_explanation': 'A#3 to C#4 spans 3 semitones, which corresponds to a minor third.'},
 {'id': 'task1_interval_identification-000001',
  'prediction_type': 'label',
  'prediction': 'octave',
  'prediction_label': 'octave',
  'prediction_explanation': 'C4 to C5 spans 12 semitones, which corresponds to a octave.'}]

In [9]:
# Step 7: run the unified evaluator
run_cmd([
    sys.executable,
    '-m',
    'benchmark.scripts.eval_predictions',
    '--gold',
    str(VIEWS_DIR / 'note_level' / 'zero_shot.jsonl'),
    '--pred',
    str(PRED_PATH),
    '--out',
    str(EVAL_PATH),
])

eval_result = json.loads(EVAL_PATH.read_text(encoding='utf-8'))
{
    'overall': eval_result['overall'],
    'tasks': list(eval_result['by_task'].keys())[:4],
}


>>> /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/asmcm-env/bin/python -m benchmark.scripts.eval_predictions --gold /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/views/note_level/zero_shot.jsonl --pred /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/oracle_predictions.jsonl --out /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/eval_predictions.json
wrote: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/eval_predictions.json
{
  "cases": 32,
  "missing": 0,
  "hit": 32,
  "accuracy": 1.0
}


{'overall': {'cases': 32, 'missing': 0, 'hit': 32, 'accuracy': 1.0},
 'tasks': ['task1_interval_identification',
  'task2_chord_identification',
  'task3_harmonic_function',
  'task4_transposition']}

In [10]:
# Step 8: annotate model outputs and inspect explanatory information
run_cmd([
    sys.executable,
    '-m',
    'benchmark.scripts.annotate_model_outputs',
    '--cases',
    str(CASES_PATH),
    '--pred',
    str(PRED_PATH),
    '--out',
    str(ANNOTATED_PATH),
    '--summary-out',
    str(SUMMARY_PATH),
])

annotated_cases = json.loads(ANNOTATED_PATH.read_text(encoding='utf-8'))
annotated_label_example = next(case for case in annotated_cases if case['task'] in LABEL_TASKS)
{
    'case_id': annotated_label_example['case_id'],
    'task': annotated_label_example['task'],
    'model_output': annotated_label_example['model_output'],
    'prediction_explanation': annotated_label_example.get('prediction_explanation', ''),
    'target_explanation': annotated_label_example.get('target_explanation', ''),
    'hit_ground_truth': annotated_label_example['hit_ground_truth'],
}


>>> /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/asmcm-env/bin/python -m benchmark.scripts.annotate_model_outputs --cases /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/all_cases.json --pred /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/oracle_predictions.jsonl --out /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/cases_with_outputs.json --summary-out /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/annotated_summary.json
wrote: /Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review/cases_with_outputs.json
cases=32 hit=32 miss=0 acc=1.0000
text_token_error_rate_avg=0.0000 text_char_error_rate_avg=0.0000 note_event_error_rate_avg=0.0000
match_status_counts= {"hit": 32}
by_task= {"task1_interval_identification": {"cases": 4, "hit": 4, "miss": 0, "acc": 1.0, "text_token_error_rate_avg": 0.0, "text_char_error_rate_avg": 0.0, "note_event_error_r

{'case_id': 'task1_interval_identification-000000',
 'task': 'task1_interval_identification',
 'model_output': 'minor_third',
 'prediction_explanation': 'A#3 to C#4 spans 3 semitones, which corresponds to a minor third.',
 'target_explanation': 'A#3 to C#4 spans 3 semitones, which corresponds to a minor third.',
 'hit_ground_truth': True}

## Results notes

Interpretation guide:

- If the standard path runs correctly, the benchmark is easy to test and easy to hand off.
- If `prediction_explanation` appears in annotated outputs without affecting the main metric, the explanatory mode is working as intended.
- For paper work, the key comparison is between structural correctness and explanatory quality.


In [11]:
# Minimal summary for copy-paste into notes or a draft
summary = {
    'workspace': str(WORKSPACE),
    'cases_built': len(cases),
    'overall_accuracy': eval_result['overall']['accuracy'],
    'label_tasks': sorted(LABEL_TASKS),
    'sequence_tasks': sorted(SEQUENCE_TASKS),
    'standard_mode_primary_fields': ['prediction_label', 'prediction_notes'],
    'explanatory_mode_optional_field': 'prediction_explanation',
}
summary


{'workspace': '/Users/meco/MBP Files/Gits/upf/ASMCM/midi_agent/tmp/notebook-benchmark-review',
 'cases_built': 32,
 'overall_accuracy': 1.0,
 'label_tasks': ['task1_interval_identification',
  'task2_chord_identification',
  'task3_harmonic_function',
  'task8_voice_leading'],
 'sequence_tasks': ['task4_transposition',
  'task5_melodic_inversion',
  'task6_retrograde',
  'task7_rhythm_scale'],
 'standard_mode_primary_fields': ['prediction_label', 'prediction_notes'],
 'explanatory_mode_optional_field': 'prediction_explanation'}

## Next steps

- Use the shell wrapper for larger runs:
  - `bash scripts/run_standard_benchmark.sh 200 agent_like none`
- Replace oracle predictions with real model outputs.
- Compare standard-mode accuracy against optional explanations for label tasks.
- Reuse the generated results in the benchmark guide, the paper draft, and the presentation.
